# Novel Approach: Modifying the Parent Paper's Method

**Last week** ([`parent_paper_1_implementation`](../parent_paper_1_implementation/)) reproduced the first parent paper. Gray, Shalan and Kadlec (IEEE CICN 2024) predict video game sales with Linear Regression and a Decision Tree. Their R² of 1.000 came from inputs that add up to the target and exist only after release.

**This week** starts from that method applied honestly to Steam, using only information available on a game's release day. It then changes one block of the pipeline at a time. Each change is a technique that I expected to improve on the parent paper's approach, taken from the second and third parent papers or added by me.

| Pipeline block | Parent paper 1 (starting point) | Technique tried | Source |
|---|---|---|---|
| Target | Sales on their raw scale | Log of owners (a log-linear model) | Parent paper 2 |
| Segments | One model for all games | Separate models for paid and free-to-play games | Parent paper 2 |
| Inputs | Raw price | Price bands | Parent paper 2 |
| Model | Linear Regression, Decision Tree | Gradient Boosting | My addition |
| Inputs | Store-page variables | Studio experience and store-page lead time | My addition |
| Inputs | Store-page variables | Sales of the game's predecessor and the studio's earlier games (analogy) | Parent paper 3 |

**Parent papers 2 and 3:**

- **Paper 2:** V. Bondarenko, J. A. Cabrera, and F. H. P. Fitzek, "Predicting Early Sales of PC Games Using Steam Wishlists and User Ratings," IEEE SmartNets 2026.
- **Paper 3:** O. Schaer, N. Kourentzes, and R. Fildes, "Analogy-based Life Cycle Forecasts with Pre-release Buzz," European Journal of Operational Research, 2026.

Neither paper's data is public (wishlists in Paper 2; weekly VGChartz sales and Google Trends in Paper 3). This notebook therefore borrows their techniques, not their variables.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.tree import DecisionTreeRegressor
from IPython.display import display

RANDOM_STATE = 42
pd.set_option("display.width", 160)

## 1. Data

The data is the [Steam Games Dataset](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset) (Kaggle, version 49), which joins Steam store pages with SteamSpy ownership estimates. After cleaning, 76,727 games released from 1997 to 2024 remain.

**Target.** SteamSpy reports owners as ranges, such as 20,000–50,000, so each game is given the midpoint of its range.

**Inputs.** Only information that exists on release day is used. Reviews, playtime and player counts are excluded.

**The data file.** `steam_features.csv.gz` holds the variables used here. They were built by [`new_analysis/notebooks/01`](../new_analysis/notebooks/01_preprocessing_and_transformation.ipynb), [`07`](../new_analysis/notebooks/07_modification_new_prerelease_variables.ipynb) and [`10`](../new_analysis/notebooks/10_modification_predecessor_analogy.ipynb). The file is saved in this folder because Kaggle no longer offers version 49 for download. `feature_groups.json` lists which variables belong to each group.

In [2]:
data = pd.read_csv("steam_features.csv.gz")
with open("feature_groups.json") as f:
    groups = json.load(f)

BASE = groups["BASE"]                                              # 42 store-page variables
STORE_PAGE = BASE + groups["STUDIO_AND_LEAD_TIME"]                 # + 7 = 49
ANALOGY = STORE_PAGE + groups["PREDECESSOR"] + groups["STUDIO_AVERAGES"]   # + 10 = 59

# Price bands (Paper 2): indicators for each band, with $0-9.99 as the reference
EDGES = [0, 10, 20, 30, 40, 60, 80, np.inf]
PRICE_BANDS = [f"price_{lo}_{hi}" if np.isfinite(hi) else f"price_{lo}_plus" for lo, hi in zip(EDGES[1:-1], EDGES[2:])]


def add_columns(frame):
    """Target on the log scale and the price-band indicators."""
    frame = frame.copy()
    frame["owners"] = (frame["owners_millions"] * 1e6).round()
    frame["log_owners"] = np.log1p(frame["owners"])
    for name, lo, hi in zip(PRICE_BANDS, EDGES[1:-1], EDGES[2:]):
        frame[name] = ((frame["price"] >= lo) & (frame["price"] < hi)).astype(int)
    return frame


print(f"{len(data):,} games | paid: {(data['price'] > 0).sum():,} | free-to-play: {(data['price'] == 0).sum():,}")
print(f"Variables: store page {len(BASE)}, + studio experience and lead time {len(STORE_PAGE)}, + analogy {len(ANALOGY)}")

76,727 games | paid: 68,380 | free-to-play: 8,347
Variables: store page 42, + studio experience and lead time 49, + analogy 59


## 2. Training, test and validation sets (70% / 20% / 10%)

The games are divided once, at random with a fixed seed, into three separate datasets, which are saved as files in this folder:

| Dataset | Share | Used for |
|---|---|---|
| `train.csv.gz` | 70% | Fitting every model |
| `test.csv.gz` | 20% | Comparing the models and deciding which changes to keep |
| `validation.csv.gz` | 10% | Unseen data: scored only once, in the last section, after the final model is chosen |

**A note on names.** This follows the course's convention, where *test* is the set checked during development and *validation* is the unseen final set. Many textbooks use the two words the other way round.

**Stratified split.** The split is stratified by owner range (under 20,000, 20,000–100,000, 100,000–1 million, and over 1 million). This gives each dataset its share of the rare big sellers.

**Validation stays sealed.** After the split, the validation set is written to its file and removed from memory. Sections 3 to 7 read only `train.csv.gz` and `test.csv.gz`. The validation file is opened again only in Section 8.

In [3]:
RANGES = [0, 20_000, 100_000, 1_000_000, np.inf]
RANGE_LABELS = ["Under 20,000", "20,000-100,000", "100,000-1 million", "Over 1 million"]
owner_range = pd.cut(data["owners_millions"] * 1e6, RANGES, labels=RANGE_LABELS, right=False)

# 10% validation first, then 20% of all games (2/9 of the remaining 90%) as the test set
rest, validation = train_test_split(data, test_size=0.10, stratify=owner_range, random_state=RANDOM_STATE)
train, test = train_test_split(rest, test_size=2 / 9, stratify=owner_range.loc[rest.index], random_state=RANDOM_STATE)

GZIP = {"method": "gzip", "mtime": 0}          # byte-identical files on every run
for name, frame in [("train", train), ("test", test), ("validation", validation)]:
    frame.to_csv(f"{name}.csv.gz", index=False, compression=GZIP)

ids = {name: set(frame["appid"]) for name, frame in [("train", train), ("test", test), ("validation", validation)]}
assert not (ids["train"] & ids["test"]) and not (ids["train"] & ids["validation"]) and not (ids["test"] & ids["validation"])
assert sum(len(v) for v in ids.values()) == len(data)
print("Saved train.csv.gz, test.csv.gz and validation.csv.gz. No game appears in more than one dataset.")

rows = {}
for name, frame in [("Training", train), ("Test", test), ("Validation", validation)]:
    ranges = owner_range.loc[frame.index].value_counts(normalize=True).reindex(RANGE_LABELS)
    rows[name] = {"Games": f"{len(frame):,}", "Share of all games": f"{len(frame) / len(data):.0%}",
                  "Free-to-play": f"{(frame['price'] == 0).mean():.1%}",
                  "Median release year": f"{frame['release_year'].median():.0f}",
                  **{f"Owners {r}": f"{v:.2%}" for r, v in ranges.items()}}
display(pd.DataFrame(rows).style.set_caption("The three datasets have the same make-up"))

del data, rest, validation          # the validation set leaves memory until Section 8

Saved train.csv.gz, test.csv.gz and validation.csv.gz. No game appears in more than one dataset.


,Training,Test,Validation
Games,"53,708","15,346","7,673"
Share of all games,70%,20%,10%
Free-to-play,10.9%,11.0%,10.6%
Median release year,2020,2020,2020
"Owners Under 20,000",69.66%,69.66%,69.66%
"Owners 20,000-100,000",19.66%,19.66%,19.67%
"Owners 100,000-1 million",9.00%,9.00%,9.01%
Owners Over 1 million,1.68%,1.68%,1.67%


### How every model is evaluated during development

Every model is fitted on the training set and scored in three ways:

1. **Random folds:** 5-fold cross-validation inside the training set.
2. **Test set:** the 20% test set.
3. **One year ahead:** each of 2022, 2023 and 2024 is predicted by a model fitted only on earlier training-set releases. This is the closest test to a real forecast.

**Main measure: R² on log owners.** Owner counts are very skewed: 70% of games have under 20,000 owners, while a few have tens of millions. On the raw scale, R² is dominated by a handful of blockbusters. Models fitted on raw owners are converted to the log scale through log(1 + max(prediction, 0)). The test set also reports R² and mean absolute error (MAE) on raw owners.

In [4]:
train = add_columns(pd.read_csv("train.csv.gz"))
test = add_columns(pd.read_csv("test.csv.gz"))
print(f"Loaded for development: training set {len(train):,} games, test set {len(test):,} games")
folds = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(train))
AHEAD_YEARS = (2022, 2023, 2024)


class PaidFreeModel:
    """Paper 2: one model for paid games and a separate one for free-to-play games."""

    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        paid = (X["price"] > 0).to_numpy()
        self.models = {seg: clone(self.model).fit(X[paid == seg], y[paid == seg]) for seg in (True, False)}
        return self

    def predict(self, X):
        paid = (X["price"] > 0).to_numpy()
        pred = np.zeros(len(X))
        for seg in (True, False):
            if (paid == seg).any():
                pred[paid == seg] = self.models[seg].predict(X[paid == seg])
        return pred


def prepare(frame, features, linear):
    """Linear models cannot use missing values: fill them with 0 and add a 'missing' indicator."""
    X = frame[features]
    if not linear:
        return X
    missing = [c for c in features if train[c].isna().any()]
    return pd.concat([X.fillna(0), X[missing].isna().astype(int).add_suffix("_missing")], axis=1)


def fit_predict(spec, fit_frame, score_frame):
    """Fit on fit_frame and return predictions of log owners for score_frame."""
    model = PaidFreeModel(spec["model"]) if spec["paid_free"] else clone(spec["model"])
    linear = isinstance(spec["model"], LinearRegression)
    X_fit, X_score = prepare(fit_frame, spec["features"], linear), prepare(score_frame, spec["features"], linear)
    if spec["target"] == "raw":
        pred = model.fit(X_fit, fit_frame["owners"] / 1e6).predict(X_score) * 1e6
        return np.log1p(np.clip(pred, 0, None))
    return model.fit(X_fit, fit_frame["log_owners"]).predict(X_score)


results, specs = [], {}


def evaluate(name, block, source, model, features, target="log", paid_free=False):
    spec = {"model": model, "features": features, "target": target, "paid_free": paid_free}
    oof = np.zeros(len(train))
    for tr, va in folds:
        oof[va] = fit_predict(spec, train.iloc[tr], train.iloc[va])
    ahead = [r2_score(train.loc[train["release_year"] == y, "log_owners"],
                      fit_predict(spec, train[train["release_year"] < y], train[train["release_year"] == y]))
             for y in AHEAD_YEARS]
    test_log = fit_predict(spec, train, test)
    row = {"Model": name, "Block changed": block, "Source": source, "Variables": len(features),
           "Random folds: R² (log)": r2_score(train["log_owners"], oof),
           "Test: R² (log)": r2_score(test["log_owners"], test_log),
           "One year ahead: R² (log)": np.mean(ahead),
           "Test: R² (owners)": r2_score(test["owners"], np.expm1(test_log)),
           "Test: MAE (owners)": mean_absolute_error(test["owners"], np.expm1(test_log))}
    results.append(row)
    print(f"{name}\n  random folds {row['Random folds: R² (log)']:.3f} | test {row['Test: R² (log)']:.3f} | "
          f"one year ahead {row['One year ahead: R² (log)']:.3f} | test MAE {row['Test: MAE (owners)']:,.0f} owners")
    specs[name] = spec

Loaded for development: training set 53,708 games, test set 15,346 games


## 3. Starting point: the parent paper's method with release-day information only

These are the parent paper's two models, Linear Regression and a Decision Tree, fitted on owners in millions, the raw scale the paper used. The inputs are the 42 store-page variables available on release day:

- price, release year, required age and number of languages
- Mac and Linux support
- number of screenshots and description length
- 15 genre tags, one yes/no column each. These are the 15 most common genres on Steam (Early Access is left out, because games lose that label after they leave it):
  - Indie, Casual, Action, Adventure, Simulation
  - Strategy, RPG, Free To Play, Sports, Racing
  - Massively Multiplayer, Violent, Gore, Education, Nudity
- 18 Steam feature tags, one yes/no column each. These are the 18 most common, leaving out tags that Valve grants or that can be added after release (for example Trading Cards, Achievements and Workshop):
  - Single-player, Steam Cloud, Full controller support, Multi-player, Partial Controller Support, PvP
  - Co-op, Steam Leaderboards, Online PvP, Shared/Split Screen, Tracked Controller Support, VR Only
  - Online Co-op, Shared/Split Screen PvP, Stats, Shared/Split Screen Co-op, Includes level editor, Captions available

In [5]:
evaluate("Baseline: Linear Regression, raw owners", "-", "Parent paper 1", LinearRegression(), BASE, target="raw")
evaluate("Baseline: Decision Tree, raw owners", "-", "Parent paper 1",
         DecisionTreeRegressor(random_state=RANDOM_STATE), BASE, target="raw")

Baseline: Linear Regression, raw owners
  random folds -25.473 | test -25.577 | one year ahead -70.648 | test MAE 198,171 owners


Baseline: Decision Tree, raw owners
  random folds -0.193 | test -0.215 | one year ahead -0.875 | test MAE 193,989 owners


**Without the leaking inputs, the parent paper's method fails.** Both models score below zero on the log scale, which means they do worse than predicting the same number for every game.

Linear Regression predicts negative owners for many small games. The Decision Tree overfits the few blockbusters. This is the starting point that the modifications below improve on.

## 4. Modifications

Each modification changes one block of the pipeline and keeps everything else the same.

### Modification 1: log target (Paper 2)

Paper 2 models the logarithm of its outcome and reads each coefficient as a percentage change. Here, Linear Regression predicts log(1 + owners) instead of owners. This keeps predictions positive and stops a few blockbusters from dominating the fit.

In [6]:
evaluate("M1: Linear Regression, log owners", "Target", "Parent paper 2", LinearRegression(), BASE)

M1: Linear Regression, log owners
  random folds 0.288 | test 0.290 | one year ahead 0.055 | test MAE 104,258 owners


### Modification 2: separate models for paid and free-to-play games (Paper 2)

Paper 2 fits paid and free-to-play games separately, because the two business models respond differently to price and ratings. Free-to-play games make up 11% of this data.

In [7]:
evaluate("M2: + separate paid and free-to-play models", "Segments", "Parent paper 2",
         LinearRegression(), BASE, paid_free=True)

M2: + separate paid and free-to-play models
  random folds 0.294 | test 0.296 | one year ahead 0.063 | test MAE 103,926 owners


### Modification 3: price bands (Paper 2)

Paper 2 compares games by price band rather than treating price as a straight line. Here, indicators for $10–19.99, $20–29.99, $30–39.99, $40–59.99, $60–79.99 and $80+ are added, with $0–9.99 as the reference. This lets the effect of price bend.

In [8]:
evaluate("M3: + price bands", "Inputs", "Parent paper 2",
                     LinearRegression(), BASE + PRICE_BANDS, paid_free=True)

M3: + price bands
  random folds 0.313 | test 0.311 | one year ahead 0.089 | test MAE 103,394 owners


### Modification 4: Gradient Boosting (my addition)

Gradient Boosting builds many small decision trees, each correcting the errors of the ones before it. Unlike Linear Regression, it can capture bends and interactions between variables, such as price mattering differently by genre. The default settings of scikit-learn's `HistGradientBoostingRegressor` are used, on log owners.

In [9]:
evaluate("M4: Gradient Boosting, log owners", "Model", "My addition",
         HistGradientBoostingRegressor(random_state=RANDOM_STATE), BASE)

M4: Gradient Boosting, log owners
  random folds 0.435 | test 0.431 | one year ahead 0.244 | test MAE 94,933 owners


### Modification 5: Gradient Boosting with separate paid and free-to-play models (Paper 2)

Next, Paper 2's paid and free-to-play split is applied to Gradient Boosting as well.

In [10]:
evaluate("M5: Gradient Boosting + separate paid and free-to-play models", "Segments", "Parent paper 2",
         HistGradientBoostingRegressor(random_state=RANDOM_STATE), BASE, paid_free=True)

M5: Gradient Boosting + separate paid and free-to-play models
  random folds 0.432 | test 0.431 | one year ahead 0.238 | test MAE 97,619 owners


The split does not help Gradient Boosting, because the trees already treat paid and free-to-play games differently. Training each model on fewer games costs a little accuracy. The split is therefore **not kept** for the following steps.

### Modification 6: studio experience and store-page lead time (my addition)

Seven variables are added, all known before release:

- **Studio experience:** how many games the developer and publisher had released before, whether this is the developer's first game, whether the game is self-published, and the number of developers.
- **Store-page lead time:** how many days before release the game's store page appeared.

In [11]:
evaluate("M6: + studio experience and lead time", "Inputs", "My addition",
         HistGradientBoostingRegressor(random_state=RANDOM_STATE), STORE_PAGE)

M6: + studio experience and lead time
  random folds 0.457 | test 0.455 | one year ahead 0.257 | test MAE 93,818 owners


### Modification 7: forecasting from the predecessor (Paper 3)

Paper 3's best model forecasts a new game from the sales of the previous game in its franchise. Here the idea is extended from franchises to every game on Steam. The predecessor is chosen in this order:

1. the previous game in the same series;
2. otherwise, the developer's previous game;
3. otherwise, the publisher's previous game.

Ten variables are added:

- the predecessor's log owners and the years since it was released, for each of the three kinds of predecessor;
- the mean and maximum log owners of the developer's and the publisher's earlier games.

When a game has no predecessor of a kind, that value is left missing. Gradient Boosting handles missing values directly.

In [12]:
evaluate("M7: + predecessor and studio sales (analogy model)", "Inputs", "Parent paper 3",
                      HistGradientBoostingRegressor(random_state=RANDOM_STATE), ANALOGY)

M7: + predecessor and studio sales (analogy model)
  random folds 0.499 | test 0.495 | one year ahead 0.312 | test MAE 92,107 owners


### Check: does the Paper 3 idea also help Linear Regression?

To see whether the gain comes from the new variables rather than from Gradient Boosting, the same variables are added to the best linear model (M3).

In [13]:
evaluate("Check: M3 + predecessor and studio sales", "Inputs", "Parent paper 3",
         LinearRegression(), ANALOGY + PRICE_BANDS, paid_free=True)

Check: M3 + predecessor and studio sales
  random folds 0.411 | test 0.400 | one year ahead 0.228 | test MAE 98,578 owners


## 5. Comparison table

In [14]:
table = pd.DataFrame(results).set_index("Model")
table.style.format({
    "Variables": "{:d}", "Random folds: R² (log)": "{:.3f}", "Test: R² (log)": "{:.3f}",
    "One year ahead: R² (log)": "{:.3f}", "Test: R² (owners)": "{:.3f}", "Test: MAE (owners)": "{:,.0f}",
}).set_caption("Every model on the same evaluations (R² on log owners unless stated)")

,Block changed,Source,Variables,Random folds: R² (log),Test: R² (log),One year ahead: R² (log),Test: R² (owners),Test: MAE (owners)
Model,,,,,,,,
"Baseline: Linear Regression, raw owners",-,Parent paper 1,42,-25.473,-25.577,-70.648,0.066,"198,171"
"Baseline: Decision Tree, raw owners",-,Parent paper 1,42,-0.193,-0.215,-0.875,-4.483,"193,989"
"M1: Linear Regression, log owners",Target,Parent paper 2,42,0.288,0.290,0.055,0.017,"104,258"
M2: + separate paid and free-to-play models,Segments,Parent paper 2,42,0.294,0.296,0.063,0.020,"103,926"
M3: + price bands,Inputs,Parent paper 2,48,0.313,0.311,0.089,0.024,"103,394"
"M4: Gradient Boosting, log owners",Model,My addition,42,0.435,0.431,0.244,0.120,"94,933"
M5: Gradient Boosting + separate paid and free-to-play models,Segments,Parent paper 2,42,0.432,0.431,0.238,0.096,"97,619"
M6: + studio experience and lead time,Inputs,My addition,49,0.457,0.455,0.257,0.145,"93,818"
M7: + predecessor and studio sales (analogy model),Inputs,Parent paper 3,59,0.499,0.495,0.312,0.158,"92,107"


## 6. Where the final model works

The final model is M7. Its one-year-ahead predictions for 2022–2024 are broken down below by the kind of predecessor a game has, and by paid or free-to-play.

In [15]:
rows = []
for year in AHEAD_YEARS:
    earlier, scored = train[train["release_year"] < year], train[train["release_year"] == year].copy()
    scored["pred"] = fit_predict(specs["M7: + predecessor and studio sales (analogy model)"], earlier, scored)
    rows.append(scored)
ahead = pd.concat(rows)


def ahead_r2(mask):
    part = ahead[mask]
    return np.mean([r2_score(g["log_owners"], g["pred"]) for _, g in part.groupby("release_year")]), len(part)


groups_out = {"All games": ahead["price"] >= 0}
for kind in ["Series predecessor", "Developer's previous game", "Publisher's previous game", "No predecessor"]:
    groups_out[kind] = ahead["analogy_type"] == kind
groups_out["Paid games"] = ahead["price"] > 0
groups_out["Free-to-play games"] = ahead["price"] == 0
breakdown = pd.DataFrame([(name, *ahead_r2(mask)) for name, mask in groups_out.items()],
                         columns=["Games", "One year ahead: R² (log)", "Games scored"]).set_index("Games")
breakdown.style.format({"One year ahead: R² (log)": "{:.3f}", "Games scored": "{:,}"})

,One year ahead: R² (log),Games scored
Games,,
All games,0.312,"20,043"
Series predecessor,0.589,"1,311"
Developer's previous game,0.383,"6,885"
Publisher's previous game,0.375,"1,368"
No predecessor,0.096,"10,479"
Paid games,0.342,"18,522"
Free-to-play games,-0.050,"1,521"


## 7. Effects as percentages (Paper 2's reporting)

Paper 2 reports each effect as a percentage change in its outcome. The same reporting is applied here to the paid and free-to-play Linear Regression models (M2), fitted on all training games. For a coefficient *b* on log owners, the effect of a one-unit increase is 100 × (e^b − 1)%. These are associations, not causes.

In [16]:
SHOWN = {"price": "Price (per $10)", "n_languages": "Each supported language",
         "n_audio_languages": "Each full-audio language", "n_screenshots": "Each screenshot",
         "mac": "Mac support", "linux": "Linux support", "cat_Multi_player": "Multi-player",
         "cat_Online_Co_op": "Online co-op", "cat_Full_controller_support": "Full controller support",
         "genre_Indie": "Indie genre tag", "genre_Massively_Multiplayer": "Massively multiplayer tag"}
effects = {}
for segment, frame in [("Paid games", train[train["price"] > 0]), ("Free-to-play games", train[train["price"] == 0])]:
    lr = LinearRegression().fit(frame[BASE], frame["log_owners"])
    coef = pd.Series(lr.coef_, index=BASE)
    coef["price"] *= 10                                   # per $10 rather than per $1
    effects[segment] = 100 * (np.exp(coef[list(SHOWN)]) - 1)
effects = pd.DataFrame(effects).rename(index=SHOWN)
effects.loc["Price (per $10)", "Free-to-play games"] = np.nan   # price is always 0 for free-to-play games
effects.style.format("{:+.1f}%", na_rep="-").set_caption("Change in expected owners, holding the other store-page variables fixed")

,Paid games,Free-to-play games
Price (per $10),+7.9%,-
Each supported language,+1.6%,+6.5%
Each full-audio language,-1.1%,-3.7%
Each screenshot,+1.4%,+1.0%
Mac support,+11.2%,+21.5%
Linux support,-2.8%,+5.8%
Multi-player,+37.8%,+42.4%
Online co-op,+44.4%,+8.3%
Full controller support,+21.3%,+11.2%
Indie genre tag,-14.2%,-20.4%


**Reading the table.**

- **Price goes the other way from Paper 2.** In Paper 2, $10 more lowered wishlist conversion by about 15%. Here, $10 more goes with about 8% more owners. The outcomes differ: Paper 2 measures how many interested players buy, while total owners also reflect how large and well funded a game is, and pricier games tend to be bigger productions.
- **Free-to-play games react more to reach.** Each supported language goes with about 6.5% more owners for free-to-play games, against about 1.6% for paid games, and Mac support with about 21% against 11%.
- **Multiplayer support goes with more owners** in both groups.

## 8. Validation (unseen data, run once)

Development is finished: M7 is the final model, chosen on the training and test sets alone. Only now is the validation file read. The models below are fitted on the training set exactly as before and scored once on the 7,673 validation games. Nothing is changed after seeing these numbers.

This notebook never used the validation games for fitting or for choosing a model. However, the ideas behind M1–M7 were first explored in my earlier notebooks ([`new_analysis/notebooks/03`–`11`](../new_analysis/notebooks/)), which used a different 80/20 split, before this validation set existed.

In [17]:
validation = add_columns(pd.read_csv("validation.csv.gz"))
print(f"Validation set: {len(validation):,} games")

FINAL = "M7: + predecessor and studio sales (analogy model)"
SCORED = ["Baseline: Linear Regression, raw owners", "Baseline: Decision Tree, raw owners",
          "M3: + price bands", "M4: Gradient Boosting, log owners", FINAL]
rows = []
for name in SCORED:
    val_log = fit_predict(specs[name], train, validation)
    rows.append({"Model": name, "Test: R² (log)": table.loc[name, "Test: R² (log)"],
                 "Validation: R² (log)": r2_score(validation["log_owners"], val_log),
                 "Validation: R² (owners)": r2_score(validation["owners"], np.expm1(val_log)),
                 "Validation: MAE (owners)": mean_absolute_error(validation["owners"], np.expm1(val_log))})
    if name == FINAL:
        validation["pred"] = val_log
validation_table = pd.DataFrame(rows).set_index("Model")
validation_table.style.format({"Test: R² (log)": "{:.3f}", "Validation: R² (log)": "{:.3f}",
                               "Validation: R² (owners)": "{:.3f}", "Validation: MAE (owners)": "{:,.0f}"}).set_caption(
    "Unseen validation set: models fitted on the training set, scored once")

Validation set: 7,673 games


,Test: R² (log),Validation: R² (log),Validation: R² (owners),Validation: MAE (owners)
Model,,,,
"Baseline: Linear Regression, raw owners",-25.577,-25.921,0.040,"195,404"
"Baseline: Decision Tree, raw owners",-0.215,-0.219,-2.240,"196,448"
M3: + price bands,0.311,0.314,0.011,"102,562"
"M4: Gradient Boosting, log owners",0.431,0.440,0.051,"95,099"
M7: + predecessor and studio sales (analogy model),0.495,0.506,0.116,"89,381"


In [18]:
by_kind = []
for kind in ["Series predecessor", "Developer's previous game", "Publisher's previous game", "No predecessor"]:
    part = validation[validation["analogy_type"] == kind]
    by_kind.append({"Games": kind, "Games scored": len(part), "Validation: R² (log)": r2_score(part["log_owners"], part["pred"])})
for label, part in [("Paid games", validation[validation["price"] > 0]), ("Free-to-play games", validation[validation["price"] == 0])]:
    by_kind.append({"Games": label, "Games scored": len(part), "Validation: R² (log)": r2_score(part["log_owners"], part["pred"])})
pd.DataFrame(by_kind).set_index("Games").style.format({"Games scored": "{:,}", "Validation: R² (log)": "{:.3f}"}).set_caption(
    "Final model on the validation set, by kind of game")

,Games scored,Validation: R² (log)
Games,,
Series predecessor,559,0.771
Developer's previous game,"2,506",0.552
Publisher's previous game,625,0.468
No predecessor,"3,983",0.363
Paid games,"6,863",0.512
Free-to-play games,810,0.441


**Reading the validation results.**

- **The results hold on unseen data.** The final model scores R² = 0.506 on the validation set, against 0.495 on the test set. Every model keeps its rank, so the choices made on the test set were not fitted to it by chance.
- **The baseline still fails** (−0.219). The parent paper's method does no better on fresh games.
- **Mean absolute error** falls from about 196,000 owners for the baseline Decision Tree to about 89,000 for the final model.
- **By kind of game, the same order appears:** 0.771 with a series predecessor, down to 0.363 with none. These are unseen games from the same years, not later years, so the scores are higher than the one-year-ahead results in Section 6.

## 9. Summary

| Step | Change | Random folds | Test | One year ahead |
|---|---|---|---|---|
| Baseline | Parent paper 1 method (Decision Tree, raw owners) | −0.193 | −0.215 | −0.875 |
| M1–M3 | Paper 2: log target, paid/free split, price bands (Linear Regression) | 0.313 | 0.311 | 0.089 |
| M4 | Gradient Boosting | 0.435 | 0.431 | 0.244 |
| M6 | Studio experience and lead time | 0.457 | 0.455 | 0.257 |
| M7 | Paper 3: predecessor and studio sales (**final model**) | **0.499** | **0.495** | **0.312** |

R² is measured on log owners. Models are fitted on the training set; "Test" is the 20% test set.

**Validation (unseen 10%).** The final model reaches R² = 0.506 on the validation set, close to its 0.495 on the test set, and the baseline stays below zero (−0.219). The improvement holds on games the models never saw during development.

**What helped.**

- **Paper 2's log target** turned a model worse than the average into a useful one.
- **Gradient Boosting** added the most on top of that.
- **Paper 3's predecessor idea** gave the largest gain from new information. It raised one-year-ahead R² from 0.257 to 0.312. It also helped Linear Regression, from 0.313 to 0.411 with random folds.
- The final model cuts the test-set mean absolute error from about 194,000 owners (baseline Decision Tree) to about 92,000.

**What did not help.** Paper 2's separate paid and free-to-play models helped Linear Regression a little but made Gradient Boosting slightly worse, so they were dropped.

**Where the final model works.** Games that continue a series can be forecast reasonably well one year ahead (R² = 0.589). Games with no earlier title from their series, developer or publisher (0.096) cannot, and neither can free-to-play games (−0.050, worse than predicting the average). Hits that nothing before release pointed to remain unpredictable.

**Limitations.**

- Owners are SteamSpy estimates given as ranges.
- Owner counts are a 2025 snapshot, so older games have had longer to sell.
- No measure of interest before release, such as wishlists or search volume, was available for most games.